# CCM Majors Survey — Cleaning Notebook  
**Student:** Hannah Raquel Melio  
**Date:** October 18, 2025

This notebook prepares the CCM Majors Survey data for analysis by renaming headers to a consistent style, selecting the fields needed for five research questions, converting checkbox responses to True/False, harmonizing demographic labels, and extracting a 1–5 interest score.



## Cleaning Goals
- Create consistent, analysis‑friendly column headers (snake_case).
- Identify and select fields required for the five questions.
- Convert checkbox responses to True/False.
- Standardize key demographics (age, gender, race_ethnicity, major).
- Extract the interest score as an integer 1–5.


## Step 0 — Setup & Load

In [1]:

import pandas as pd
import numpy as np
import re
from pathlib import Path

# Input and output live next to the notebook for a simple workflow
DEFAULT_FILENAME = "majors_survey_cleaned.csv"
path = Path.cwd() / DEFAULT_FILENAME

# If you prefer a different file, update this variable or pick a file when prompted
if not path.exists():
    try:
        import tkinter as tk
        from tkinter import filedialog
        tk.Tk().withdraw()
        picked = filedialog.askopenfilename(title="Select your survey CSV file", filetypes=[("CSV files","*.csv"), ("All files","*.*")])
        if picked:
            path = Path(picked)
    except Exception:
        pass

if not path.exists():
    raise FileNotFoundError(f"Could not find '{DEFAULT_FILENAME}' in {Path.cwd()}. Place your CSV next to this notebook or select a file when prompted.")

df_raw = pd.read_csv(path)
df = df_raw.copy()

def to_snake(s: str) -> str:
    s = s.strip()
    s = re.sub(r"[\/\-]+", "_", s)
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_]+", "", s)
    return s.lower()

# 1) Normalize headers to snake_case
df.columns = [to_snake(c) for c in df.columns]
df.head(3)


,timestamp,age,gender,race_ethnicity,major,interest_1to5,please_explain_your_answer_to_the_question_above_why_or_why_not_would_you_be_interested_in_taking_another_computing_class,how_did_you_hear_about_county_college_of_morris_ccm_web_site,how_did_you_hear_about_county_college_of_morris_social_media,how_did_you_hear_about_county_college_of_morris_community_event,...,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_transfer_credits_back_to_hs_degree_sharetime_challenger_program,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_career_advancement,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_career_change,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_professional_development,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_job_displacement,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_relocation,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_to_keep_current_in_tech_industry,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_it_industry_certifications,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_financial,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_personal_enrichment
0,2024/09/09 12:33:58 PM AST,21-24,Man,White/Caucasian,Mechanical Engineering Technologies,2.0,Quite difficult and I do not see myself using ...,True,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024/09/09 6:00:11 PM AST,18 and younger,Man,Hispanic or Latino;White/Caucasian;Multi-Racial,Cis Game Development Option,NaN,NaN,False,False,True,...,False,False,False,False,False,False,False,False,False,True
2,2024/09/10 9:00:12 AM AST,18 and younger,Man,Hispanic or Latino;Asian,Computer Science,NaN,NaN,False,False,False,...,False,False,False,True,False,False,True,False,False,False



## Step 1 — Column Audit
Identify likely columns for the five research questions using keyword patterns.


In [2]:

def find_cols(patterns):
    pat = re.compile("|".join(patterns), flags=re.I)
    return [c for c in df.columns if pat.search(c)]

discovery = {
    "timestamp_like": find_cols([r"\btime\b", r"timestamp"]),
    "reasons_like": find_cols([r"reason", r"why.*class", r"motivation", r"take.*comput"]),
    "heard_like": find_cols([r"how_did_you_hear", r"hear", r"heard", r"find.*out"]),
    "events_like": find_cols([r"prior_to_applying_to_college", r"open.*house", r"info.*session", r"workshop", r"orientation"]),
    "interest_like": [c for c in df.columns if re.search(r"interested|interest|another_computing_class|1_to_5", c, flags=re.I)],
    "age_like": find_cols([r"\bage\b", r"age_group"]),
    "gender_like": find_cols([r"gender", r"\bsex\b"]),
    "major_like": find_cols([r"what_degree_program_are_you_currently_enrolled_in", r"major", r"program_of_study"]),
    "race_ethnicity_like": find_cols([r"race_ethnicity", r"race.*eth", r"raceethnicity"]),
}
discovery


{'timestamp_like': ['timestamp'],
 'reasons_like': ['please_explain_your_answer_to_the_question_above_why_or_why_not_would_you_be_interested_in_taking_another_computing_class'],
 'heard_like': ['how_did_you_hear_about_county_college_of_morris_ccm_web_site',
  'how_did_you_hear_about_county_college_of_morris_social_media',
  'how_did_you_hear_about_county_college_of_morris_community_event',
  'how_did_you_hear_about_county_college_of_morris_family_member_or_friend',
  'how_did_you_hear_about_county_college_of_morris_current_ccm_student',
  'how_did_you_hear_about_county_college_of_morris_ccm_alumni',
  'how_did_you_hear_about_county_college_of_morris_high_school_teacher',
  'how_did_you_hear_about_county_college_of_morris_high_school_counselor',
  'how_did_you_hear_about_county_college_of_morris_in_app_advertisement',
  'how_did_you_hear_about_county_college_of_morris_employer',
  'how_did_you_hear_about_county_college_of_morris_billboard',
  'how_did_you_hear_about_county_college_of_mo


## Step 2 — Explicit Selections
Select the exact columns used for analysis:
- Motivations: `what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_*`
- How students heard about CCM: `how_did_you_hear_about_county_college_of_morris_*`
- Events/activities: `prior_to_applying_to_college_did_you_participate_with_the_department_of_information_technologies_if_at_all_*`
- Interest scale: `on_a_scale_of_1_to_5...`
- Demographics: `age`, `gender`, `race_ethnicity`, `what_degree_program_are_you_currently_enrolled_in`


In [3]:

def cols_starting_with(prefix):
    return [c for c in df.columns if c.startswith(prefix)]

timestamp_col = "timestamp" if "timestamp" in df.columns else None
interest_col = [c for c in df.columns if c.startswith("on_a_scale_of_1_to_5")]
interest_col = interest_col[0] if interest_col else None
interest_free = [c for c in df.columns if c.startswith("please_explain_your_answer_to_the_question_above")]
interest_free = interest_free[0] if interest_free else None

age_col = "age" if "age" in df.columns else None
gender_col = "gender" if "gender" in df.columns else None
race_eth_col = "race_ethnicity" if "race_ethnicity" in df.columns else None
major_col = "what_degree_program_are_you_currently_enrolled_in" if "what_degree_program_are_you_currently_enrolled_in" in df.columns else None

mot_cols = cols_starting_with("what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_")
heard_cols = cols_starting_with("how_did_you_hear_about_county_college_of_morris_")
events_cols = cols_starting_with("prior_to_applying_to_college_did_you_participate_with_the_department_of_information_technologies_if_at_all_")

len(mot_cols), len(heard_cols), len(events_cols)


(12, 14, 0)


## Step 3 — Value Normalization
- Convert checkbox responses to True/False.
- Extract interest score as an integer 1–5.
- Standardize demographics to consistent text.


In [4]:

YES = {"yes", "y", "true", "t", "1", 1, True, "checked", "selected"}
NO  = {"no", "n", "false", "f", "0", 0, False, "not selected", "unselected", "unchecked", ""}

def to_bool(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if s in YES: return True
    if s in NO:  return False
    return val

dfc = df.copy()
for c in mot_cols + heard_cols + events_cols:
    dfc[c] = dfc[c].apply(to_bool)

def parse_1_5(v):
    if pd.isna(v): return np.nan
    m = re.search(r"([1-5])", str(v))
    return int(m.group(1)) if m else pd.to_numeric(v, errors="coerce")

if interest_col:
    dfc["interest_1to5"] = dfc[interest_col].apply(parse_1_5)

# Demographics to normalized text columns (temporary names)
if gender_col:
    dfc["gender_clean"] = dfc[gender_col].astype(str).str.strip().str.replace("_", " ", regex=False).str.title()
if age_col:
    dfc["age_clean"] = dfc[age_col].astype(str).str.strip().replace({"nan": np.nan})
if race_eth_col:
    dfc["race_ethnicity_clean"] = (
        dfc[race_eth_col].astype(str).str.replace(r"\s{2,}", " ", regex=True).str.strip().replace({"nan": np.nan})
    )
if major_col:
    dfc["major_clean"] = (
        dfc[major_col].astype(str).str.strip().replace({"nan": np.nan}).str.replace(r"\s+", " ", regex=True).str.title()
    )

dfc.head(3)


,timestamp,age,gender,race_ethnicity,major,interest_1to5,please_explain_your_answer_to_the_question_above_why_or_why_not_would_you_be_interested_in_taking_another_computing_class,how_did_you_hear_about_county_college_of_morris_ccm_web_site,how_did_you_hear_about_county_college_of_morris_social_media,how_did_you_hear_about_county_college_of_morris_community_event,...,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_professional_development,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_job_displacement,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_relocation,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_to_keep_current_in_tech_industry,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_it_industry_certifications,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_financial,what_motivated_you_to_seek_a_computing_degree_certificate_at_ccm_personal_enrichment,gender_clean,age_clean,race_ethnicity_clean
0,2024/09/09 12:33:58 PM AST,21-24,Man,White/Caucasian,Mechanical Engineering Technologies,2.0,Quite difficult and I do not see myself using ...,True,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Man,21-24,White/Caucasian
1,2024/09/09 6:00:11 PM AST,18 and younger,Man,Hispanic or Latino;White/Caucasian;Multi-Racial,Cis Game Development Option,NaN,NaN,False,False,True,...,False,False,False,False,False,False,True,Man,18 and younger,Hispanic or Latino;White/Caucasian;Multi-Racial
2,2024/09/10 9:00:12 AM AST,18 and younger,Man,Hispanic or Latino;Asian,Computer Science,NaN,NaN,False,False,False,...,True,False,False,True,False,False,False,Man,18 and younger,Hispanic or Latino;Asian



## Step 4 — Final Column Set
Assemble only the fields needed for analysis and finalize names.


In [5]:

keep_cols = []
if timestamp_col: keep_cols.append(timestamp_col)
if age_col: keep_cols.append("age_clean")
if gender_col: keep_cols.append("gender_clean")
if race_eth_col: keep_cols.append("race_ethnicity_clean")
if major_col: keep_cols.append("major_clean")
if interest_col: keep_cols.append("interest_1to5")
if interest_free: keep_cols.append(interest_free)

keep_cols += heard_cols + events_cols + mot_cols
keep_cols = [c for c in keep_cols if c in dfc.columns]

final_df = dfc[keep_cols].copy()

# Remove the temporary "_clean" suffixes for the final output
final_df.columns = [re.sub(r"_clean$", "", c) for c in final_df.columns]
final_df.shape, final_df.head(3)


((197, 31),
                     timestamp             age gender  \
 0  2024/09/09 12:33:58 PM AST           21-24    Man   
 1   2024/09/09 6:00:11 PM AST  18 and younger    Man   
 2   2024/09/10 9:00:12 AM AST  18 and younger    Man   
 
                                     race_ethnicity  \
 0                                  White/Caucasian   
 1  Hispanic or Latino;White/Caucasian;Multi-Racial   
 2                         Hispanic or Latino;Asian   
 
   please_explain_your_answer_to_the_question_above_why_or_why_not_would_you_be_interested_in_taking_another_computing_class  \
 0  Quite difficult and I do not see myself using ...                                                                          
 1                                                NaN                                                                          
 2                                                NaN                                                                          
 
   how_did_you_hear_ab


## Step 5 — Save Output
Save the ready-to-analyze dataset next to the notebook.


In [6]:

out_path = Path("majors_survey_cleaned.csv")
final_df.to_csv(out_path, index=False)
out_path.as_posix()


'majors_survey_cleaned.csv'


### Summary
- Headers standardized to snake_case for reliable selection.
- Required columns identified for motivations, sources, events, interest, and demographics.
- Checkbox values normalized to True/False.
- Demographic labels standardized to consistent, readable text.
- Interest captured as an integer 1–5.
- Final dataset saved as **majors_survey_cleaned.csv** with concise column names.


## References

###### County College of Morris. (2024). *Majors Survey Results – Fall 2024* [Dataset]. Department of Information Technologies, County College of Morris.

###### Pandas Development Team. (2024). *pandas* [Computer software]. Zenodo. https://doi.org/10.5281/zenodo.3509134

###### Python Software Foundation. (2024). *Python: A programming language* [Computer software]. https://www.python.org/

###### ChatGPT, OpenAI. (2025). *Survey cleaning assistance and notebook generation*. Accessed October 18, 2025.